# Handover Optimization using GRU
Implementation based on: "Sequence-Based Deep Learning for Handover Optimization"

In [5]:
# Imports and Configuration
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Configuration
WINDOW_SIZE = 10
BATCH_SIZE = 32
EPOCHS = 50

np.random.seed(42)
tf.random.set_seed(42)

## 1. Data Preparation

In [6]:
# Load and preprocess data
df = pd.read_csv("processed_dataset.csv")
df = df.sort_values(['SessionID', 'ElapsedTime']).reset_index(drop=True)

# Handover type: 0=No HO, 1=Normal, 2=Ping-Pong
curr = df[['Node', 'CellID']]
prev = curr.shift(1)
two_back = curr.shift(2)
df['Handover_type'] = np.where((curr == prev).all(axis=1), 0,
                    np.where((curr == two_back).all(axis=1), 2, 1))
df.loc[:1, 'Handover_type'] = 0

# Time of Stay (ToS): time until next handover
next_cell = curr.shift(-1)
time_diff = df['ElapsedTime'].shift(-1) - df['ElapsedTime']
df['ToS'] = np.where((curr == next_cell).all(axis=1), time_diff, 0).astype(float)
df['ToS'] = df['ToS'].fillna(0)

# Signal slopes for detection
dt = df['ElapsedTime'].diff()
df['RSRP_slopes'] = df['Level'].diff() / dt
df['SNR_slopes'] = df['SNR'].diff() / dt
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Bearing for direction detection
lat, lon = np.radians(df['Latitude']), np.radians(df['Longitude'])
prev_lat = df.groupby('SessionID')['Latitude'].shift(1).apply(np.radians)
prev_lon = df.groupby('SessionID')['Longitude'].shift(1).apply(np.radians)
dLon = lon - prev_lon
y = np.sin(dLon) * np.cos(lat)
x = np.cos(prev_lat) * np.sin(lat) - np.sin(prev_lat) * np.cos(lat) * np.cos(dLon)
df['Bearing'] = ((np.degrees(np.arctan2(y, x)) + 360) % 360).fillna(0.0)

print(f"Handover types: {df['Handover_type'].value_counts().to_dict()}")

Handover types: {0: 17551, 1: 7433, 2: 5941}


In [7]:
# Feature normalization (keep original for Algorithm 2)
df_original = df.copy()

feature_columns = ['Level', 'Qual', 'SNR', 'CQI', 'SecondCell_RSRP', 'SecondCell_SNR',
                    'NRxLev1', 'NQual1', 'Speed', 'DL_bitrate', 'UL_bitrate', 'BANDWIDTH']

df[feature_columns] = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(0.0)
scaler = StandardScaler()
df[feature_columns] = scaler.fit_transform(df[feature_columns])

## 2. Create Sequences & Train/Test Split

In [19]:
def create_sequences(df_part, features, window_size):
    #Create sliding window sequences
    data = df_part[features].to_numpy(dtype=np.float32)
    n = len(df_part) - window_size
    X = np.array([data[i:i+window_size] for i in range(n)])
    y_type = df_part['Handover_type'].to_numpy()[window_size:].astype(np.int32)
    y_tos = df_part['ToS'].to_numpy()[window_size:].astype(np.float32)
    return X, y_type, y_tos

# Split: 70% train, 15% val, 15% test
N = len(df)
df_train = df.iloc[:int(0.70*N)].reset_index(drop=True)
df_val = df.iloc[int(0.70*N):int(0.85*N)].reset_index(drop=True)
df_test = df.iloc[int(0.85*N):].reset_index(drop=True)

X_train, y_train_type, y_train_tos = create_sequences(df_train, feature_columns, WINDOW_SIZE)
X_val, y_val_type, y_val_tos = create_sequences(df_val, feature_columns, WINDOW_SIZE)
X_test, y_test_type, y_test_tos = create_sequences(df_test, feature_columns, WINDOW_SIZE)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (21637, 10, 12), Val: (4629, 10, 12), Test: (4629, 10, 12)


In [ ]:
# Oversample ping-pong class (2x) to balance training
unique_classes, class_counts = np.unique(y_train_type, return_counts=True)
max_count = class_counts.max()
oversample = {0: 1.0, 1: 1.0, 2: 2.0}  # 2x boost for ping-pong

X_list, y_type_list, y_tos_list = [], [], []
for cls in unique_classes:
    idx = np.where(y_train_type == cls)[0]
    X_list.append(X_train[idx])
    y_type_list.append(y_train_type[idx])
    y_tos_list.append(y_train_tos[idx])
    
    needed = int(max_count * oversample.get(cls, 1.0)) - len(idx)
    if needed > 0:
        dup = np.random.choice(idx, size=needed, replace=True)
        X_list.append(X_train[dup])
        y_type_list.append(y_train_type[dup])
        y_tos_list.append(y_train_tos[dup])

X_train = np.concatenate(X_list)
y_train_type = np.concatenate(y_type_list)
y_train_tos = np.concatenate(y_tos_list)

# Shuffle
perm = np.random.permutation(len(X_train))
X_train, y_train_type, y_train_tos = X_train[perm], y_train_type[perm], y_train_tos[perm]

print(f"After oversampling: {len(X_train)} samples")

After oversampling: 45868 samples


## 3. Build and Train GRU Model

In [15]:
# GRU model with dual output: ToS prediction + Handover classification
inputs = Input(shape=(WINDOW_SIZE, len(feature_columns)))
x = GRU(128, return_sequences=True)(inputs)
x = Dropout(0.3)(x)
x = GRU(64)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

tos_output = Dense(1, name='ToS_out')(x)
class_output = Dense(3, activation='softmax', name='Class_out')(x)

model = Model(inputs=inputs, outputs=[tos_output, class_output])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'ToS_out': 'mse', 'Class_out': 'sparse_categorical_crossentropy'},
    loss_weights={'ToS_out': 0.5, 'Class_out': 5.0},
    metrics={'Class_out': 'accuracy'}
)

history = model.fit(
    X_train, {'ToS_out': y_train_tos, 'Class_out': y_train_type},
    validation_data=(X_val, {'ToS_out': y_val_tos, 'Class_out': y_val_type}),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
    callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
)

Epoch 1/50
1434/1434 - 35s - loss: 89.9538 - ToS_out_loss: 170.6186 - Class_out_loss: 0.9289 - Class_out_accuracy: 0.5717 - val_loss: 6.1840 - val_ToS_out_loss: 2.2253 - val_Class_out_loss: 1.0143 - val_Class_out_accuracy: 0.4753 - 35s/epoch - 25ms/step
Epoch 2/50
1434/1434 - 35s - loss: 89.9538 - ToS_out_loss: 170.6186 - Class_out_loss: 0.9289 - Class_out_accuracy: 0.5717 - val_loss: 6.1840 - val_ToS_out_loss: 2.2253 - val_Class_out_loss: 1.0143 - val_Class_out_accuracy: 0.4753 - 35s/epoch - 25ms/step
Epoch 2/50
1434/1434 - 23s - loss: 89.4262 - ToS_out_loss: 170.8574 - Class_out_loss: 0.7995 - Class_out_accuracy: 0.6528 - val_loss: 5.8806 - val_ToS_out_loss: 2.2635 - val_Class_out_loss: 0.9498 - val_Class_out_accuracy: 0.5273 - 23s/epoch - 16ms/step
Epoch 3/50
1434/1434 - 23s - loss: 89.4262 - ToS_out_loss: 170.8574 - Class_out_loss: 0.7995 - Class_out_accuracy: 0.6528 - val_loss: 5.8806 - val_ToS_out_loss: 2.2635 - val_Class_out_loss: 0.9498 - val_Class_out_accuracy: 0.5273 - 23s/ep

In [ ]:
# Evaluate model
pred_tos, pred_class = model.predict(X_test)
pred_class_labels = np.argmax(pred_class, axis=1)
pred_tos_flat = pred_tos.flatten()

print(f"ToS RMSE: {np.sqrt(mean_squared_error(y_test_tos, pred_tos_flat)):.4f}\n")
print(classification_report(y_test_type, pred_class_labels, digits=4,
                            target_names=['No Handover', 'Normal', 'Ping-Pong']))

145/145 [==============================] - 2s 8ms/step
ToS RMSE: 3.1901
ToS RMSE: 3.1901

              precision    recall  f1-score   support

 No Handover     0.9156    0.6881    0.7857      3578
      Normal     0.3173    0.4704    0.3789       574
   Ping-Pong     0.3590    0.8197    0.4994       477

    accuracy                         0.6747      4629
   macro avg     0.5306    0.6594    0.5547      4629
weighted avg     0.7840    0.6747    0.7058      4629


              precision    recall  f1-score   support

 No Handover     0.9156    0.6881    0.7857      3578
      Normal     0.3173    0.4704    0.3789       574
   Ping-Pong     0.3590    0.8197    0.4994       477

    accuracy                         0.6747      4629
   macro avg     0.5306    0.6594    0.5547      4629
weighted avg     0.7840    0.6747    0.7058      4629



## 4. Algorithm 1: Ping-Pong Detection

In [17]:
# Thresholds
ToS_th = 1.2
rsrp_th = df['RSRP_slopes'].dropna().abs().quantile(0.85)
snr_th = df['SNR_slopes'].dropna().abs().quantile(0.85)

# Get test slopes (last value in each window)
def get_last_values(df_part, col, ws):
    vals = df_part[col].fillna(0.0).to_numpy()
    return np.array([vals[i + ws - 1] for i in range(len(df_part) - ws)])

test_rsrp_slopes = get_last_values(df_test, 'RSRP_slopes', WINDOW_SIZE)
test_snr_slopes = get_last_values(df_test, 'SNR_slopes', WINDOW_SIZE)

# Detection: short stay + signal oscillation OR model predicts PP
short = pred_tos_flat < ToS_th
osc = (np.abs(test_rsrp_slopes) > rsrp_th) | (np.abs(test_snr_slopes) > snr_th)
model_pp = (pred_class_labels == 2)
is_pp = (short & osc) | model_pp

# Metrics
actual_pp = (y_test_type == 2)
total_pp = np.sum(actual_pp)
tp = np.sum(is_pp & actual_pp)
prec = tp / np.sum(is_pp) if np.sum(is_pp) > 0 else 0
rec = tp / total_pp if total_pp > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print("=== Algorithm 1: Detection ===")
print(f"Actual PP: {total_pp}, Detected: {np.sum(is_pp)}, True Positives: {tp}")
print(f"Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

=== Algorithm 1: Detection ===
Actual PP: 477, Detected: 1167, True Positives: 410
Precision: 0.3513, Recall: 0.8595, F1: 0.4988


## 5. Algorithm 2: Handover Avoidance

In [18]:
# Prepare original (unnormalized) test data
df_orig_test = df_original.iloc[int(0.85 * len(df_original)):].reset_index(drop=True)
test_rsrp = get_last_values(df_orig_test, 'Level', WINDOW_SIZE)

# Get bearings
bearing = df_orig_test['Bearing'].fillna(0.0).to_numpy()
n = len(df_orig_test) - WINDOW_SIZE
bear_curr = np.array([bearing[i + WINDOW_SIZE - 1] for i in range(n)])
bear_prev = np.array([bearing[i + WINDOW_SIZE - 2] for i in range(n)])

# Thresholds
level_th = -110
ToS_unnec_th = 1.35

# Bearing difference (handle 360° wrap)
bear_diff = np.abs(bear_curr - bear_prev)
bear_diff = np.minimum(bear_diff, 360 - bear_diff)

# Avoidance conditions
safe = test_rsrp > level_th
moving_away = bear_diff > 45
unnec = safe & moving_away & (pred_tos_flat < ToS_unnec_th)

avoid = is_pp | unnec
execute = ~avoid

# Metrics
n_avoided = np.sum(avoid)
correct_avoided = np.sum(avoid & actual_pp)

print("=== Algorithm 2: Avoidance ===")
print(f"Total Avoided: {n_avoided}, Correctly Avoided (PP): {correct_avoided}")
print(f"Executed Handovers: {np.sum(execute)}")
print(f"HO Reduction: {n_avoided / len(y_test_type) * 100:.2f}%")
print(f"PP Reduction: {correct_avoided / total_pp * 100:.2f}%")

=== Algorithm 2: Avoidance ===
Total Avoided: 1921, Correctly Avoided (PP): 457
Executed Handovers: 2708
HO Reduction: 41.50%
PP Reduction: 95.81%


## 6. Save Model & Artifacts

In [19]:
import joblib
import os

# Create artifacts directory
os.makedirs('artifacts', exist_ok=True)

# Save model
model.save('model/handover_model_final.keras')

# Save scaler for preprocessing new data
joblib.dump(scaler, 'artifacts/scaler.joblib')

# Save configuration and thresholds
config = {
    'WINDOW_SIZE': WINDOW_SIZE,
    'feature_columns': feature_columns,
    'ToS_th': ToS_th,
    'ToS_unnec_th': ToS_unnec_th,
    'level_th': level_th,
    'rsrp_th': float(rsrp_th),
    'snr_th': float(snr_th),
    'bearing_th': 45
}
joblib.dump(config, 'artifacts/config.joblib')

print("Saved:")
print("  - model/handover_model_final.keras")
print("  - artifacts/scaler.joblib")
print("  - artifacts/config.joblib")

Saved:
  - model/handover_model_final.keras
  - artifacts/scaler.joblib
  - artifacts/config.joblib
